# IdiomX Demo Notebook

This notebook builds a local demo for the final IdiomX system.

The demo uses two layers:

1. **Task 1 — Idiom Detection**  
   Checks whether the input sentence is likely idiomatic or literal.

2. **Task 4 — Idiom Meaning Retrieval**  
   If the sentence is idiomatic, retrieves the most likely idiom and returns its English, Arabic, and French meanings when available.

The same structure is designed to work locally and later inside a Hugging Face Space.

In [4]:
# [0.1] Install required packages only if missing

import importlib
import subprocess
import sys


def ensure_package(pkg_name, import_name=None):
    """
    Install package only if missing.
    """
    if import_name is None:
        import_name = pkg_name

    try:
        importlib.import_module(import_name)
        print(f"{pkg_name} already installed")

    except ImportError:
        print(f"Installing {pkg_name} ...")

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                pkg_name
            ]
        )


# Core packages for demo
ensure_package("faiss-cpu", "faiss")
ensure_package("rank-bm25")
ensure_package("sentence-transformers")
ensure_package("transformers")
ensure_package("pyarrow")

print("Environment ready.")

Installing faiss-cpu ...
Installing rank-bm25 ...
Installing sentence-transformers ...


C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers already installed
pyarrow already installed
Environment ready.


In [1]:
# [1.1] Setup dynamic artifact paths

from pathlib import Path
import os

# Current notebook location
NOTEBOOK_DIR = Path.cwd()

# Main project directory
# If notebook is inside notebooks/, go one level up
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

# Shared demo artifacts folder
DEMO_ARTIFACT_DIR = PROJECT_DIR / "demo_artifacts"

# Task 1 model path
TASK1_MODEL_DIR = DEMO_ARTIFACT_DIR / "task1_idiom_detector" / "roberta"

# Task 4 retrieval artifact path
TASK4_ARTIFACT_DIR = DEMO_ARTIFACT_DIR / "task4_meaning_retrieval"

print("Project directory:", PROJECT_DIR)
print("Demo artifacts:", DEMO_ARTIFACT_DIR)
print("Task 1 model:", TASK1_MODEL_DIR)
print("Task 4 artifacts:", TASK4_ARTIFACT_DIR)

Project directory: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX
Demo artifacts: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts
Task 1 model: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts\task1_idiom_detector\roberta
Task 4 artifacts: C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts\task4_meaning_retrieval


## 2. Verify Demo Artifacts

Check that Task 1 detector files and Task 4 retrieval files exist before loading the demo pipeline.

In [2]:
# [2.1] Verify required artifact files

required_files = {
    "Task 1 config": TASK1_MODEL_DIR / "config.json",
    "Task 1 model": TASK1_MODEL_DIR / "model.safetensors",
    "Task 1 tokenizer": TASK1_MODEL_DIR / "tokenizer.json",

    "Task 4 retrieval bank": TASK4_ARTIFACT_DIR / "final_retrieval_bank.parquet",
    "Task 4 FAISS index": TASK4_ARTIFACT_DIR / "final_faiss.index",
    "Task 4 BM25 tokens": TASK4_ARTIFACT_DIR / "bm25_corpus_tokens.pkl",
    "Task 4 config": TASK4_ARTIFACT_DIR / "model_config.json"
}

for name, path in required_files.items():
    # Print whether each required file exists
    print(f"{name}: {'OK' if path.exists() else 'MISSING'} -> {path}")

Task 1 config: OK -> C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts\task1_idiom_detector\roberta\config.json
Task 1 model: OK -> C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts\task1_idiom_detector\roberta\model.safetensors
Task 1 tokenizer: OK -> C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts\task1_idiom_detector\roberta\tokenizer.json
Task 4 retrieval bank: OK -> C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts\task4_meaning_retrieval\final_retrieval_bank.parquet
Task 4 FAISS index: OK -> C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts\task4_meaning_retrieval\final_faiss.index
Task 4 BM25 tokens: OK -> C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts\task4_meaning_retrieval\bm25_corpus_tokens.pkl
Task 4 config: OK -> C:\Users\ayman\Documents\IdiomX\github_idiomX\IdiomX\demo_artifacts\task4_meaning_retrieval\model_config.json


In [5]:
# [3.1] Import required libraries

import json
import pickle
import numpy as np
import pandas as pd
import torch
import faiss

from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, CrossEncoder

print("Libraries imported successfully.")

Libraries imported successfully.


In [6]:
# [3.2] Load Task 4 configuration

with open(TASK4_ARTIFACT_DIR / "model_config.json", "r") as f:
    model_config = json.load(f)

EMBEDDING_MODEL_NAME = model_config["embedding_model"]
RERANKER_MODEL_NAME = model_config["reranker_model"]

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Reranker model:", RERANKER_MODEL_NAME)

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2


## 4. Load Idiom Detection Model (Task 1)

Load the fine-tuned RoBERTa idiom detector used as the first layer of the demo.

In [7]:
# [4.1] Load Task 1 idiom detector

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

task1_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        TASK1_MODEL_DIR
    )
)

task1_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        TASK1_MODEL_DIR
    )
    .to(device)
)

task1_model.eval()

print(
    "Task 1 detector loaded on",
    device
)

Task 1 detector loaded on cuda


In [15]:
# [5.1] Load retrieval bank and indexes

# Load retrieval bank
df_bank = pd.read_parquet(
    TASK4_ARTIFACT_DIR /
    "final_retrieval_bank.parquet"
)

# Load dense embeddings
retrieval_embeddings = np.load(
    TASK4_ARTIFACT_DIR /
    "final_retrieval_embeddings.npy"
)

# Load FAISS index
faiss_index = faiss.read_index(
    str(
        TASK4_ARTIFACT_DIR /
        "final_faiss.index"
    )
)

# Rebuild BM25 from the same retrieval bank to avoid row mismatch
bm25_corpus_tokens = [
    str(text).lower().split()
    for text in df_bank["input_text"]
]

bm25 = BM25Okapi(
    bm25_corpus_tokens
)

print("Retrieval bank loaded:", len(df_bank), "entries")
print("FAISS index rows:", faiss_index.ntotal)
print("BM25 rows:", len(bm25_corpus_tokens))

Retrieval bank loaded: 67518 entries
FAISS index rows: 67518
BM25 rows: 67518


In [9]:
# [6.1] Load embedding model and reranker

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

reranker_model = CrossEncoder(
    RERANKER_MODEL_NAME
)

print("Embedding model loaded:", EMBEDDING_MODEL_NAME)
print("Reranker loaded:", RERANKER_MODEL_NAME)

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [11]:
# [7.1] Idiom detection function with readable labels

LABEL_MAP = {
    "LABEL_0": "Literal",
    "LABEL_1": "Idiomatic"
}


def detect_idiom_usage(text):

    # Tokenize input
    inputs = task1_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        k: v.to(device)
        for k,v in inputs.items()
    }

    # Inference
    with torch.no_grad():
        outputs = task1_model(**inputs)

    probs = torch.softmax(
        outputs.logits,
        dim=-1
    )[0].cpu().numpy()

    pred_id = int(
        np.argmax(probs)
    )

    raw_label = (
        task1_model
        .config
        .id2label[pred_id]
    )

    label = LABEL_MAP.get(
        raw_label,
        raw_label
    )

    return {
        "label": label,
        "confidence":
            float(probs[pred_id])
    }


# sanity tests
tests = [
    "She spilled the tea about the party.",
    "She spilled the tea on the floor."
]

for t in tests:
    print("\n", t)
    print(
        detect_idiom_usage(t)
    )


 She spilled the tea about the party.
{'label': 'Idiomatic', 'confidence': 0.996272087097168}

 She spilled the tea on the floor.
{'label': 'Idiomatic', 'confidence': 0.8431667685508728}


## 8. Two-Layer IdiomX Demo Pipeline

Pipeline:

1. Detect whether the input appears idiomatic or ambiguous  
2. Retrieve idiom meaning candidates using Hybrid + Reranker

In [23]:
# [8.1] Final retrieval function for demo

def retrieve_meanings(
    query,
    top_k=3,
    retrieve_k=20
):

    # Dense query embedding
    q_emb = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    # FAISS retrieval
    scores, idxs = faiss_index.search(
        q_emb.astype("float32"),
        retrieve_k
    )

    dense_candidates = (
        df_bank
        .iloc[idxs[0]]
        .copy()
    )

    # BM25 lexical scores
    bm25_scores = bm25.get_scores(
        query.lower().split()
    )

    dense_candidates["bm25_score"] = [
        bm25_scores[i]
        for i in idxs[0]
    ]

    # Hybrid score
    dense_candidates["hybrid_score"] = (
        0.65 * scores[0]
        +
        0.35 *
        (
            dense_candidates["bm25_score"] /
            dense_candidates["bm25_score"].max()
        )
    )

    # Candidate pairs for reranker
    pairs=[]

    for _,row in dense_candidates.iterrows():

        candidate_text = (
            str(row["idiom_canonical"])
            + " | " +
            str(row["idiom_canonical_meaning"])
        )

        pairs.append(
            (query,candidate_text)
        )

    rerank_scores = reranker_model.predict(
        pairs
    )

    dense_candidates["rerank_score"] = (
        rerank_scores
    )

    results = (
        dense_candidates
        .sort_values("rerank_score", ascending=False)
        .drop_duplicates("idiom_canonical")
        .head(top_k)
    )

    return results

In [24]:
# [8.2] Two-layer demo pipeline

def idiomx_pipeline(query):

    det = detect_idiom_usage(query)

    print("\n"+"="*95)
    print("INPUT:",query)

    print("\nLayer 1 — Idiom Detection")
    print(
        f"{det['label']} "
        f"(confidence={det['confidence']:.3f})"
    )

    if det["confidence"] < 0.90:
        print(
            "Ambiguous reading possible."
        )
    
    if det["label"] == "Literal":
        print(
            "\nNote: The detector suggests this sentence is literal. "
            "The meanings below are shown only as related idiom suggestions."
    )

    print("\nLayer 2 — Meaning Retrieval")

    results = retrieve_meanings(
        query,
        top_k=3
    )

    for rank,(_,row) in enumerate(
        results.iterrows(),
        1
    ):

        print("\n"+"-"*80)
        print(f"Rank {rank}")
        print(
            "Idiom:",
            row["idiom_canonical"]
        )

        print(
            "Meaning EN:",
            row["idiom_canonical_meaning"]
        )

        if pd.notna(
            row.get(
                "idiom_canonical_meaning_arabic"
            )
        ):
            print(
                "Meaning AR:",
                row[
                  "idiom_canonical_meaning_arabic"
                ]
            )

In [25]:
idiomx_pipeline(
    "She spilled the tea about the party"
)

idiomx_pipeline(
    "She spilled the tea on the floor"
)


INPUT: She spilled the tea about the party

Layer 1 — Idiom Detection
Idiomatic (confidence=0.998)

Layer 2 — Meaning Retrieval

--------------------------------------------------------------------------------
Rank 1
Idiom: spill the tea
Meaning EN: To disclose or reveal gossip or personal, often secret, information about someone.
Meaning AR: كشف أو إفشاء الشائعات أو المعلومات الشخصية، غالبًا السرية، عن شخص ما.

--------------------------------------------------------------------------------
Rank 2
Idiom: tea party
Meaning EN: An event or interaction characterized by politeness, civility, and avoidance of conflict, typically implying a delicate or overly courteous atmosphere.
Meaning AR: حدث أو تفاعل يتسم باللباقة والتهذيب وتجنب الصراع، وعادة ما يشير إلى جو دقيق أو مجامل للغاية.

--------------------------------------------------------------------------------
Rank 3
Idiom: clock the tea
Meaning EN: To notice or discover the truth or hidden facts, especially about gossip or secret info

In [26]:
# [9.1] Final polished demo pipeline

def idiomx_pipeline(query, top_k=3):

    # Run Task 1 detector first
    det = detect_idiom_usage(query)

    print("\n" + "=" * 95)
    print("INPUT:", query)

    print("\nLayer 1 — Idiom Detection")
    print(
        f"{det['label']} "
        f"(confidence={det['confidence']:.3f})"
    )

    # Warn user if detector thinks input is literal
    if det["label"] == "Literal":
        print(
            "\nNote: The detector suggests this sentence is literal. "
            "The meanings below are shown only as related idiom suggestions."
        )

    # Warn if confidence is not very high
    elif det["confidence"] < 0.90:
        print(
            "\nNote: The detector is not fully confident. "
            "This sentence may be ambiguous."
        )

    print("\nLayer 2 — Meaning Retrieval")

    # Retrieve meanings using final retrieval pipeline
    results = retrieve_meanings(
        query,
        top_k=top_k
    )

    for rank, (_, row) in enumerate(results.iterrows(), 1):

        print("\n" + "-" * 80)
        print(f"Rank {rank}")

        print("Idiom:", row["idiom_canonical"])

        print("\nMeaning EN:")
        print(row["idiom_canonical_meaning"])

        # Arabic meaning
        if pd.notna(row.get("idiom_canonical_meaning_arabic")):
            print("\nMeaning AR:")
            print(row["idiom_canonical_meaning_arabic"])

        # French meaning if available
        if pd.notna(row.get("idiom_canonical_meaning_french")):
            print("\nMeaning FR:")
            print(row["idiom_canonical_meaning_french"])

        # Reranker score
        if pd.notna(row.get("rerank_score")):
            print(
                "\nRerank Score:",
                round(row["rerank_score"], 4)
            )

In [27]:
# [9.2] Test polished demo

idiomx_pipeline(
    "She spilled the tea about the party"
)

idiomx_pipeline(
    "She spilled the tea on the floor"
)


INPUT: She spilled the tea about the party

Layer 1 — Idiom Detection
Idiomatic (confidence=0.998)

Layer 2 — Meaning Retrieval

--------------------------------------------------------------------------------
Rank 1
Idiom: spill the tea

Meaning EN:
To disclose or reveal gossip or personal, often secret, information about someone.

Meaning AR:
كشف أو إفشاء الشائعات أو المعلومات الشخصية، غالبًا السرية، عن شخص ما.

Meaning FR:
Révéler des potins ou des informations personnelles, souvent secrètes, sur quelqu’un.

Rerank Score: 2.1491

--------------------------------------------------------------------------------
Rank 2
Idiom: tea party

Meaning EN:
An event or interaction characterized by politeness, civility, and avoidance of conflict, typically implying a delicate or overly courteous atmosphere.

Meaning AR:
حدث أو تفاعل يتسم باللباقة والتهذيب وتجنب الصراع، وعادة ما يشير إلى جو دقيق أو مجامل للغاية.

Rerank Score: -2.3361

------------------------------------------------------------

In [29]:
# [10.1] Interactive demo loop

while True:

    # Read user input
    query = input(
        "\nEnter an idiom or sentence (type quit to stop): "
    )

    # Stop condition
    if query.strip().lower() in ["quit", "exit", "q"]:
        print("Demo stopped.")
        break

    # Run full two-layer IdiomX pipeline
    idiomx_pipeline(
        query,
        top_k=3
    )


Enter an idiom or sentence (type quit to stop): he spill the bean

INPUT: he spill the bean

Layer 1 — Idiom Detection
Idiomatic (confidence=1.000)

Layer 2 — Meaning Retrieval

--------------------------------------------------------------------------------
Rank 1
Idiom: spill the beans

Meaning EN:
To reveal secret or confidential information, often unintentionally or prematurely.

Meaning AR:
كشف معلومات سرية أو خاصة، غالبًا بدون قصد أو قبل الوقت المناسب.

Meaning FR:
Révéler des informations secrètes ou confidentielles, souvent involontairement ou prématurément.

Rerank Score: 5.4929

--------------------------------------------------------------------------------
Rank 2
Idiom: spill the tea

Meaning EN:
To disclose or reveal gossip or personal, often secret, information about someone.

Meaning AR:
كشف أو إفشاء الشائعات أو المعلومات الشخصية، غالبًا السرية، عن شخص ما.

Meaning FR:
Révéler des potins ou des informations personnelles, souvent secrètes, sur quelqu’un.

Rerank Score: -